# ASTRA - Module 1 (OGM) + Module 2 (DLC) Debug Notebook

Notebook này là bản debug tương tác cho repo hiện tại:

- Module 1: YOLOE-26X bbox marking qua `models.image_generator.generate_bbox_image`.
- Module 2: Depth-Anything-V2 heatmap qua `compute_depth_cue` và `render_depth_image`.
- Prompt debug: dùng `models.prompt_v2.build_prompt`.

Notebook không clone repo, không cài package, và không còn phụ thuộc GroundingDINO. Hãy cài dependencies bằng `requirements.txt` trước khi chạy.

## Cell 1 - Setup local repo

Cell này tìm repo root, chuyển working directory về root, inject root vào `sys.path`, rồi đọc cấu hình pipeline hiện tại.

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "models").is_dir() and (path / "config").is_dir() and (path / "notebooks").is_dir():
            return path
    raise RuntimeError(f"Could not find ASTRA repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

from config.pipeline_config import (
    DEBUG_DIR,
    DEPTH_COLORMAP,
    DEPTH_DIFF_THRESHOLD,
    DEPTH_MODEL_SIZE,
    DET_CONF_THRESHOLD,
    EXTRACTION_FILE,
    IMAGE_DIR,
)

DEBUG_NOTEBOOK_DIR = REPO_ROOT / "output" / "debug_notebook"

print(f"Repo root: {REPO_ROOT}")
print(f"EXTRACTION_FILE: {EXTRACTION_FILE}")
print(f"IMAGE_DIR: {IMAGE_DIR}")
print(f"DEBUG_DIR: {DEBUG_DIR}")
print(f"DEBUG_NOTEBOOK_DIR: {DEBUG_NOTEBOOK_DIR}")
print(f"DET_CONF_THRESHOLD: {DET_CONF_THRESHOLD}")
print(f"DEPTH_MODEL_SIZE: {DEPTH_MODEL_SIZE}")
print(f"DEPTH_COLORMAP: {DEPTH_COLORMAP}")

## Cell 2 - Import ASTRA helpers

Các helper này là cùng lớp API đang được `scripts/debug_pipeline.py`, `step1_bbox_images.py`, và `step2_depth_images.py` dùng.

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import torch
from PIL import Image

from models.image_generator import (
    compute_depth_cue,
    depth_relation_text,
    generate_bbox_image,
    load_depth_model,
    load_yoloe_model,
    render_depth_image,
    should_run_yoloe,
)
from models.prompt_v2 import build_prompt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_SAMPLES = 2
SEED = 42

print(f"DEVICE: {DEVICE}")
print(f"NUM_SAMPLES: {NUM_SAMPLES}")
print(f"SEED: {SEED}")

## Cell 3 - Load extraction records and choose diverse samples

Chọn mẫu đa dạng theo viewer/non-viewer, confidence cao/thấp, và hallucination để dễ kiểm tra fallback.

In [ ]:
random.seed(SEED)

extraction_path = Path(EXTRACTION_FILE)
if not extraction_path.exists():
    raise FileNotFoundError(f"Extraction file not found: {extraction_path}")

with extraction_path.open("r", encoding="utf-8") as f:
    all_records = json.load(f)

groups = {
    "high_conf_viewer": [],
    "high_conf_nonviewer": [],
    "low_conf": [],
    "hallucination": [],
}

for record in all_records:
    is_viewer = bool(record.get("O2_is_viewer", False))
    confidence = float(record.get("confidence", 0.0) or 0.0)
    hallucinated = bool(record.get("O1_hallucinated", False) or record.get("O2_hallucinated", False))

    if hallucinated:
        groups["hallucination"].append(record)
    elif confidence < 0.6:
        groups["low_conf"].append(record)
    elif is_viewer:
        groups["high_conf_viewer"].append(record)
    else:
        groups["high_conf_nonviewer"].append(record)

selected = []
per_group = max(1, NUM_SAMPLES // max(1, len(groups)))
for records in groups.values():
    selected.extend(records[:per_group])

if len(selected) < NUM_SAMPLES:
    selected_ids = {str(record.get("id")) for record in selected}
    for record in all_records:
        sid = str(record.get("id"))
        if sid not in selected_ids:
            selected.append(record)
            selected_ids.add(sid)
        if len(selected) >= NUM_SAMPLES:
            break

random.shuffle(selected)
selected = selected[:NUM_SAMPLES]

print(f"Loaded {len(all_records)} records")
print(f"Selected {len(selected)} samples:")
for record in selected:
    sid = record.get("id")
    confidence = float(record.get("confidence", 0.0) or 0.0)
    hallucinated = bool(record.get("O1_hallucinated", False) or record.get("O2_hallucinated", False))
    print(
        f"  id={sid}: viewer={record.get('O2_is_viewer')}, "
        f"conf={confidence:.2f}, halluc={hallucinated}, image={record.get('image', '')}"
    )

## Cell 4 - Image path helper

Dùng `IMAGE_DIR` từ config hiện tại và thử cả `.jpg`/`.png`. Nếu chưa có image folder local, pipeline sẽ ghi lỗi từng sample và chạy tiếp.

In [ ]:
IMAGE_DIR_CANDIDATES = [
    Path(IMAGE_DIR),
    REPO_ROOT / "data" / "images" / "relevant_images",
    REPO_ROOT / "data" / "images",
    REPO_ROOT / "dataset" / "images" / "relevant_images",
    REPO_ROOT / "dataset" / "images" / "COCO2017",
    REPO_ROOT / "dataset" / "images",
]

# Keep order while removing duplicates.
IMAGE_DIR_CANDIDATES = list(dict.fromkeys(path.resolve() for path in IMAGE_DIR_CANDIDATES))


def find_image_path(image_name: str) -> Path | None:
    if not image_name:
        return None

    names = [
        image_name,
        image_name.replace(".jpg", ".png"),
        image_name.replace(".png", ".jpg"),
    ]
    for base in IMAGE_DIR_CANDIDATES:
        for name in names:
            path = base / name
            if path.exists():
                return path
    return None


print("Image search dirs:")
for base in IMAGE_DIR_CANDIDATES:
    marker = "exists" if base.exists() else "missing"
    print(f"  [{marker}] {base}")

missing = []
for record in selected:
    image_name = record.get("image", "")
    image_path = find_image_path(image_name)
    print(f"{image_name} -> {image_path}")
    if image_path is None:
        missing.append(image_name)

if missing:
    print(f"WARNING: {len(missing)}/{len(selected)} selected images were not found in configured/fallback image dirs")


## Cell 5 - Load models

Model load được bọc `try/except` để notebook vẫn chạy và ghi metadata khi thiếu weight/dependency/GPU.

In [ ]:
yoloe_model = None
try:
    print(f"Loading YOLOE-26X on {DEVICE}...")
    yoloe_model = load_yoloe_model(DEVICE)
    print("YOLOE-26X loaded")
except Exception as exc:
    print(f"WARNING: Could not load YOLOE-26X: {exc}")

depth_model = None
try:
    print(f"Loading Depth-Anything-V2 ({DEPTH_MODEL_SIZE}) on {DEVICE}...")
    depth_model = load_depth_model(DEPTH_MODEL_SIZE, DEVICE)
    print("Depth-Anything-V2 loaded")
except Exception as exc:
    print(f"WARNING: Could not load Depth-Anything-V2: {exc}")

## Cell 6 - Run Module 1 + Module 2 debug pipeline

Output được lưu vào `output/debug_notebook/{idx}_{id}/`: ảnh gốc, ảnh bbox, ảnh depth, prompt và metadata.

In [ ]:
DEBUG_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

results = []

for idx, record in enumerate(selected):
    sid = str(record.get("id", idx))
    image_name = record.get("image", "")
    sample_dir = DEBUG_NOTEBOOK_DIR / f"{idx}_{sid}"
    sample_dir.mkdir(parents=True, exist_ok=True)

    o1_name = record.get("Object", {}).get("O1", "")
    o2_name = record.get("Object", {}).get("O2", "")
    is_viewer = bool(record.get("O2_is_viewer", False))
    confidence = float(record.get("confidence", 0.0) or 0.0)
    hallucinated = bool(record.get("O1_hallucinated", False) or record.get("O2_hallucinated", False))

    print(f"\n[{idx + 1}/{len(selected)}] id={sid} | image={image_name}")
    print(f"  O1={o1_name} | O2={o2_name} | viewer={is_viewer} | conf={confidence:.2f}")

    meta = {
        "id": sid,
        "image": image_name,
        "question": record.get("question", ""),
        "options": record.get("options", []),
        "answer": record.get("answer", ""),
        "o1": o1_name,
        "o2": o2_name,
        "o2_is_viewer": is_viewer,
        "confidence": confidence,
        "o1_hallucinated": bool(record.get("O1_hallucinated", False)),
        "o2_hallucinated": bool(record.get("O2_hallucinated", False)),
    }

    image_path = find_image_path(image_name)
    if image_path is None:
        print(f"  ERROR: image not found under IMAGE_DIR: {image_name}")
        meta.update({
            "error": "image_not_found",
            "gating_pass": False,
            "marks_ok": False,
            "depth_ok": False,
        })
        with (sample_dir / "meta.json").open("w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        results.append(meta)
        continue

    image = Image.open(image_path).convert("RGB")
    image.save(sample_dir / "image_orig.jpg", quality=95)

    gating_pass = should_run_yoloe(record)
    meta["gating_pass"] = gating_pass

    marks_ok = False
    box_info = {
        "marks_ok": False,
        "box_o1": None,
        "box_o2": None,
        "o1_name": o1_name,
        "o2_name": None if is_viewer else o2_name,
        "o2_is_viewer": is_viewer,
        "conf_o1": 0.0,
        "conf_o2": 0.0,
    }
    marked_image = image.copy()

    if gating_pass and yoloe_model is not None:
        marked_image, box_info = generate_bbox_image(
            image=image,
            record=record,
            yoloe_model=yoloe_model,
            device=DEVICE,
            det_threshold=DET_CONF_THRESHOLD,
        )
        marks_ok = bool(box_info.get("marks_ok", False))
    elif not gating_pass:
        print(f"  [M1] SKIP: confidence/hallucination gating failed (hallucinated={hallucinated})")
    else:
        print("  [M1] SKIP: YOLOE model is unavailable")

    marked_image.save(sample_dir / "image1_bbox.jpg", quality=95)
    meta["marks_ok"] = marks_ok
    meta["box_info"] = box_info
    print(f"  [M1] marks_ok={marks_ok}, box_o1={box_info.get('box_o1')}, box_o2={box_info.get('box_o2')}")

    depth_ok = False
    depth_o1 = 0.0
    depth_o2 = 0.0
    relation_text = ""
    depth_image = image.copy()

    if marks_ok and depth_model is not None:
        try:
            depth_map, depth_o1, depth_o2 = compute_depth_cue(
                image=image,
                box_info=box_info,
                depth_model=depth_model,
                device=DEVICE,
                model_size=DEPTH_MODEL_SIZE,
            )
            depth_image = render_depth_image(
                depth_map=depth_map,
                box_info=box_info,
                depth_o1=depth_o1,
                depth_o2=depth_o2,
                is_viewer=is_viewer,
                colormap=DEPTH_COLORMAP,
            )
            relation_text = depth_relation_text(
                depth_o1,
                depth_o2,
                box_info.get("o1_name", o1_name),
                box_info.get("o2_name") or o2_name,
                DEPTH_DIFF_THRESHOLD,
            )
            depth_ok = True
        except Exception as exc:
            print(f"  [M2] ERROR: depth failed: {exc}")
    elif not marks_ok:
        print("  [M2] SKIP: marks_ok=False")
    else:
        print("  [M2] SKIP: depth model is unavailable")

    depth_image.save(sample_dir / "image2_depth.jpg", quality=85)
    print(f"  [M2] depth_ok={depth_ok}, d1={depth_o1:.3f}, d2={depth_o2:.3f}")

    prompt = build_prompt(
        record=record,
        marks_ok=marks_ok and depth_ok,
        depth_o1=depth_o1,
        depth_o2=depth_o2,
    )
    (sample_dir / "prompt.txt").write_text(prompt, encoding="utf-8")

    meta.update({
        "depth_ok": depth_ok,
        "depth_o1": round(float(depth_o1), 4),
        "depth_o2": round(float(depth_o2), 4),
        "relation_text": relation_text,
        "prompt": prompt,
        "output_dir": str(sample_dir),
    })

    with (sample_dir / "meta.json").open("w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    results.append(meta)

summary = {
    "total": len(results),
    "gating_pass": sum(1 for item in results if item.get("gating_pass")),
    "m1_marks_ok": sum(1 for item in results if item.get("marks_ok")),
    "m2_depth_ok": sum(1 for item in results if item.get("depth_ok")),
    "image_missing": sum(1 for item in results if item.get("error") == "image_not_found"),
}

print("\nSummary")
for key, value in summary.items():
    print(f"  {key}: {value}")
print(f"Output: {DEBUG_NOTEBOOK_DIR}")

## Cell 7 - Visualize debug outputs

Hiển thị 3 ảnh cho mỗi sample: original, bbox-marked, depth heatmap.

In [ ]:
def load_meta(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


sample_dirs = sorted(path for path in DEBUG_NOTEBOOK_DIR.iterdir() if path.is_dir())
MAX_VIS = min(5, len(sample_dirs))

for sample_dir in sample_dirs[:MAX_VIS]:
    meta_path = sample_dir / "meta.json"
    if not meta_path.exists():
        continue

    meta = load_meta(meta_path)
    image_paths = [
        sample_dir / "image_orig.jpg",
        sample_dir / "image1_bbox.jpg",
        sample_dir / "image2_depth.jpg",
    ]
    titles = ["Original", "Module 1: bbox", "Module 2: depth"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        f"id={meta.get('id')} | O1={meta.get('o1')} | O2={meta.get('o2')} | "
        f"viewer={meta.get('o2_is_viewer')} | gating={meta.get('gating_pass')} | "
        f"marks_ok={meta.get('marks_ok')} | depth_ok={meta.get('depth_ok')}",
        fontsize=11,
    )

    for ax, image_path, title in zip(axes, image_paths, titles):
        ax.set_title(title)
        ax.axis("off")
        if image_path.exists():
            ax.imshow(mpimg.imread(image_path))
        else:
            ax.text(0.5, 0.5, "missing", ha="center", va="center")

    plt.tight_layout()
    plt.show()

## Cell 8 - Inspect one prompt

Dùng cell này để xem prompt ASTRA sinh ra cho một sample debug.

In [ ]:
if results:
    prompt_sample = next((item for item in results if item.get("prompt")), results[0])
    print(f"Prompt sample id={prompt_sample.get('id')}")
    print("=" * 80)
    print(prompt_sample.get("prompt", ""))
else:
    print("No results yet. Run Cell 6 first.")